<h1 style="color:DodgerBlue">Индивидуальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

**Система работы с различными видами счетов-фактур (Invoice)**

<h2 style="color:DodgerBlue">Вариант задания:</h2>

**Вариант № 10**

<h2 style="color:DodgerBlue">Описание проекта:</h2>

В данном проекте реализуется базовый класс `Invoice`, предназначенный для хранения информации о счёте-фактуре и его позициях. На основе базового класса создаются производные классы `GoodsInvoice`, `ServiceInvoice` и `CombinedInvoice`.

Проект демонстрирует основные принципы объектно-ориентированного программирования: **наследование, инкапсуляцию и полиморфизм**. Для демонстрации полиморфизма методы `CalculateTotal()`, `AddLine()` и `RemoveLine()` в производных классах переопределяются с помощью ключевого слова `override`.

При разработке также используются идеи, изученные в предыдущей теме: разделение ответственности классов и расширение поведения без изменения базового класса.


<h2 style="color:DodgerBlue">Реализация:</h2>

Сначала создадим класс `LineItem`, который описывает отдельную позицию счёта-фактуры. Он содержит название позиции, количество, цену за единицу, а также дополнительные поля для даты поставки и причины отмены.

Затем создадим базовый класс `Invoice`. Он содержит номер счёта, дату выдачи, общую сумму и список позиций. Методы `CalculateTotal()`, `AddLine()` и `RemoveLine()` объявлены как `virtual`, чтобы производные классы могли изменить их поведение.


In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

public class LineItem
{
    public string Name { get; set; }
    public int Quantity { get; set; }
    public decimal UnitPrice { get; set; }
    public DateTime? SupplyDate { get; set; }
    public string CancellationReason { get; set; }

    public LineItem(string name, int quantity, decimal unitPrice)
    {
        Name = name;
        Quantity = quantity;
        UnitPrice = unitPrice;
    }

    public decimal GetAmount()
    {
        return Quantity * UnitPrice;
    }

    public override string ToString()
    {
        return $"{Name}: {Quantity} x {UnitPrice:N2} = {GetAmount():N2} руб.";
    }
}

public class Invoice
{
    public string InvoiceNumber { get; set; }
    public DateTime IssueDate { get; set; }
    public decimal TotalAmount { get; protected set; }

    protected List<LineItem> Lines { get; set; } = new List<LineItem>();

    public Invoice(string invoiceNumber, DateTime issueDate)
    {
        InvoiceNumber = invoiceNumber;
        IssueDate = issueDate;
    }

    public virtual decimal CalculateTotal()
    {
        TotalAmount = 0;

        foreach (LineItem line in Lines)
        {
            TotalAmount += line.GetAmount();
        }

        return TotalAmount;
    }

    public virtual void AddLine(LineItem lineItem)
    {
        Lines.Add(lineItem);
        CalculateTotal();
    }

    public virtual void RemoveLine(LineItem lineItem)
    {
        Lines.Remove(lineItem);
        CalculateTotal();
    }

    public virtual void PrintInfo()
    {
        Console.WriteLine("Номер фактуры: " + InvoiceNumber);
        Console.WriteLine("Дата выдачи: " + IssueDate.ToString("dd.MM.yyyy"));
        Console.WriteLine("Общая сумма: " + CalculateTotal().ToString("N2") + " руб.");
    }
}

// Проверка базового класса
var testInvoice = new Invoice("TEST-001", new DateTime(2026, 9, 12));
var testLine = new LineItem("Тестовый товар", 2, 1000);

testInvoice.AddLine(testLine);

Console.WriteLine("=== ПРОВЕРКА БАЗОВОГО КЛАССА ===");
testInvoice.PrintInfo();
Console.WriteLine($"Позиция: {testLine}");


The below script needs to be able to find the current output cell; this is an easy method to get it.

=== ПРОВЕРКА БАЗОВОГО КЛАССА ===
Номер фактуры: TEST-001
Дата выдачи: 12.09.2026
Общая сумма: 2 000,00 руб.
Позиция: Тестовый товар: 2 x 1 000,00 = 2 000,00 руб.


### 1. Товарная фактура - `GoodsInvoice`

Класс `GoodsInvoice` наследуется от `Invoice` и добавляет атрибут `SupplyDate`.

Метод `AddLine()` переопределяется: кроме обычного добавления позиции, для неё устанавливается дата поставки. Это показывает, что один и тот же метод может выполнять дополнительную работу в производном классе.


In [2]:
public class GoodsInvoice : Invoice
{
    public DateTime SupplyDate { get; set; }

    public GoodsInvoice(string invoiceNumber, DateTime issueDate, DateTime supplyDate)
        : base(invoiceNumber, issueDate)
    {
        SupplyDate = supplyDate;
    }

    public override void AddLine(LineItem lineItem)
    {
        lineItem.SupplyDate = SupplyDate;
        base.AddLine(lineItem);

        Console.WriteLine($"Товарная позиция \"{lineItem.Name}\" добавлена. " +
                          $"Дата поставки: {SupplyDate:dd.MM.yyyy}");
    }

    public override void PrintInfo()
    {
        Console.WriteLine("ТОВАРНАЯ ФАКТУРА");
        base.PrintInfo();
        Console.WriteLine($"Дата поставки: {SupplyDate:dd.MM.yyyy}");
    }
}

// Проверка GoodsInvoice
var goodsTest = new GoodsInvoice(
    "G-TEST",
    new DateTime(2026, 9, 12),
    new DateTime(2026, 9, 15));

goodsTest.AddLine(new LineItem("Ноутбук", 1, 75000));
goodsTest.AddLine(new LineItem("Мышь", 2, 1500));
goodsTest.PrintInfo();


Товарная позиция "Ноутбук" добавлена. Дата поставки: 15.09.2026
Товарная позиция "Мышь" добавлена. Дата поставки: 15.09.2026
ТОВАРНАЯ ФАКТУРА
Номер фактуры: G-TEST
Дата выдачи: 12.09.2026
Общая сумма: 78 000,00 руб.
Дата поставки: 15.09.2026


### 2. Услуговая фактура — `ServiceInvoice`

Класс `ServiceInvoice` добавляет атрибут `ServiceDate`.

Метод `RemoveLine()` переопределяется. При удалении позиции дополнительно выводится причина аннулирования услуги. Таким образом, поведение базового метода расширяется.


In [59]:
public class ServiceInvoice : Invoice
{
    public DateTime ServiceDate { get; set; }

    public ServiceInvoice(string invoiceNumber, DateTime issueDate, DateTime serviceDate)
        : base(invoiceNumber, issueDate)
    {
        ServiceDate = serviceDate;
    }

    public override void RemoveLine(LineItem lineItem)
    {
        string reason = "Услуга отменена клиентом";
        lineItem.CancellationReason = reason;

        Console.WriteLine($"Удаление услуги \"{lineItem.Name}\". Причина: {reason}");

        base.RemoveLine(lineItem);
    }

    public override void PrintInfo()
    {
        Console.WriteLine(" УСЛУГОВАЯ ФАКТУРА ");
        base.PrintInfo();
        Console.WriteLine($"Дата оказания услуги: {ServiceDate:dd.MM.yyyy}");
    }
}

// Проверка ServiceInvoice
var serviceTest = new ServiceInvoice(
    "S-TEST",
    new DateTime(2026, 9, 12),
    new DateTime(2026, 9, 13));

var serviceLine1 = new LineItem("Установка ПО", 1, 5000);
var serviceLine2 = new LineItem("Консультация", 2, 2500);

serviceTest.AddLine(serviceLine1);
serviceTest.AddLine(serviceLine2);

Console.WriteLine("До удаления:");
serviceTest.PrintInfo();

serviceTest.RemoveLine(serviceLine2);

Console.WriteLine("После удаления:");
serviceTest.PrintInfo();


До удаления:
 УСЛУГОВАЯ ФАКТУРА 
Номер фактуры: S-TEST
Дата выдачи: 12.09.2026
Общая сумма: 10 000,00 руб.
Дата оказания услуги: 13.09.2026
Удаление услуги "Консультация". Причина: Услуга отменена клиентом
После удаления:
 УСЛУГОВАЯ ФАКТУРА 
Номер фактуры: S-TEST
Дата выдачи: 12.09.2026
Общая сумма: 5 000,00 руб.
Дата оказания услуги: 13.09.2026


### 3. Комбинированная фактура — `CombinedInvoice`

Третий класс объединяет товары и услуги.

Он содержит атрибут `ReturnAllowed`, показывающий, разрешён ли возврат. Дополнительно используется `ReturnAmount` - сумма фактического возврата. Метод `CalculateTotal()` переопределяется: если возврат разрешён, сумма возврата вычитается из общей суммы.

Такой вариант позволяет наглядно показать, что один и тот же метод `CalculateTotal()` может работать по-разному в зависимости от типа объекта.


In [60]:
public class CombinedInvoice : Invoice
{
    public bool ReturnAllowed { get; set; }
    public decimal ReturnAmount { get; set; }

    public CombinedInvoice(
        string invoiceNumber,
        DateTime issueDate,
        bool returnAllowed)
        : base(invoiceNumber, issueDate)
    {
        ReturnAllowed = returnAllowed;
        ReturnAmount = 0;
    }

    public override decimal CalculateTotal()
    {
        decimal baseTotal = Lines.Sum(line => line.GetAmount());

        if (ReturnAllowed)
            TotalAmount = Math.Max(0, baseTotal - ReturnAmount);
        else
            TotalAmount = baseTotal;

        return TotalAmount;
    }

    public override void PrintInfo()
    {
        Console.WriteLine("=== КОМБИНИРОВАННАЯ ФАКТУРА ===");
        base.PrintInfo();
        Console.WriteLine(
            $"Возврат разрешён: {(ReturnAllowed ? "Да" : "Нет")}");
        Console.WriteLine($"Сумма возврата: {ReturnAmount:N2} руб.");
    }
}

// Проверка CombinedInvoice
var combinedTest = new CombinedInvoice(
    "C-TEST",
    new DateTime(2026, 9, 12),
    true);

combinedTest.AddLine(new LineItem("Оборудование", 1, 40000));
combinedTest.AddLine(new LineItem("Настройка", 1, 10000));

combinedTest.ReturnAmount = 5000;

combinedTest.PrintInfo();
Console.WriteLine($"Итог с учётом возврата: {combinedTest.CalculateTotal():N2} руб.");


=== КОМБИНИРОВАННАЯ ФАКТУРА ===
Номер фактуры: C-TEST
Дата выдачи: 12.09.2026
Общая сумма: 45 000,00 руб.
Возврат разрешён: Да
Сумма возврата: 5 000,00 руб.
Итог с учётом возврата: 45 000,00 руб.


## Демонстрация полиморфизма

Создадим объекты всех трёх производных классов и поместим их в одну коллекцию типа `Invoice`.

Несмотря на то, что переменная имеет тип базового класса, при вызове виртуальных методов будет выполняться реализация соответствующего производного класса. Это и есть **полиморфизм**.


In [61]:
using System;


var goodsInvoice = new GoodsInvoice(
    "G-001",
    new DateTime(2026, 9, 12),
    new DateTime(2026, 9, 15));

var serviceInvoice = new ServiceInvoice(
    "S-001",
    new DateTime(2026, 9, 12),
    new DateTime(2026, 9, 13));

var combinedInvoice = new CombinedInvoice(
    "C-001",
    new DateTime(2026, 9, 12),
    true);

var laptop = new LineItem("Ноутбук", 1, 75000);
var mouse = new LineItem("Компьютерная мышь", 2, 1500);
goodsInvoice.AddLine(laptop);
goodsInvoice.AddLine(mouse);

var installation = new LineItem("Установка ПО", 1, 5000);
var consultation = new LineItem("Консультация", 2, 2500);
serviceInvoice.AddLine(installation);
serviceInvoice.AddLine(consultation);

var equipment = new LineItem("Компьютерное оборудование", 1, 40000);
var setup = new LineItem("Настройка оборудования", 1, 10000);
combinedInvoice.AddLine(equipment);
combinedInvoice.AddLine(setup);

combinedInvoice.ReturnAmount = 5000;

serviceInvoice.RemoveLine(consultation);

Console.WriteLine("ПОЗИЦИИ ДОБАВЛЕНЫ И УДАЛЕНЫ");
Console.WriteLine("Проверка завершена.");



public class LineItem
{
    public string Name { get; set; }
    public int Quantity { get; set; }
    public decimal Price { get; set; }
    public DateTime SupplyDate { get; set; }
    public string CancellationReason { get; set; }

    public LineItem(string name, int quantity, decimal price)
    {
        Name = name;
        Quantity = quantity;
        Price = price;
    }

    public decimal Total => Quantity * Price;
}

public class Invoice
{
    public string InvoiceNumber { get; set; }
    public DateTime IssueDate { get; set; }
    protected List<LineItem> Lines { get; } = new();

    public Invoice(string invoiceNumber, DateTime issueDate)
    {
        InvoiceNumber = invoiceNumber;
        IssueDate = issueDate;
    }

    public virtual void AddLine(LineItem lineItem) => Lines.Add(lineItem);
    public virtual void RemoveLine(LineItem lineItem) => Lines.Remove(lineItem);

    public virtual void PrintInfo()
    {
        Console.WriteLine($"Фактура №{InvoiceNumber} от {IssueDate:dd.MM.yyyy}");
    }
}

public class GoodsInvoice : Invoice
{
    public DateTime SupplyDate { get; set; }

    public GoodsInvoice(string invoiceNumber, DateTime issueDate, DateTime supplyDate)
        : base(invoiceNumber, issueDate)
    {
        SupplyDate = supplyDate;
    }

    public override void AddLine(LineItem lineItem)
    {
        lineItem.SupplyDate = SupplyDate;
        base.AddLine(lineItem);
        Console.WriteLine($"Товарная позиция \"{lineItem.Name}\" добавлена. Дата поставки: {SupplyDate:dd.MM.yyyy}");
    }

    public override void PrintInfo()
    {
        Console.WriteLine(" ТОВАРНАЯ ФАКТУРА");
        base.PrintInfo();
        Console.WriteLine($"Дата поставки: {SupplyDate:dd.MM.yyyy}");
    }
}

public class ServiceInvoice : Invoice
{
    public DateTime ServiceDate { get; set; }

    public ServiceInvoice(string invoiceNumber, DateTime issueDate, DateTime serviceDate)
        : base(invoiceNumber, issueDate)
    {
        ServiceDate = serviceDate;
    }

    public override void RemoveLine(LineItem lineItem)
    {
        string reason = "Услуга отменена клиентом";
        lineItem.CancellationReason = reason;
        Console.WriteLine($"Удаление услуги \"{lineItem.Name}\". Причина: {reason}");
        base.RemoveLine(lineItem);
    }

    public override void PrintInfo()
    {
        Console.WriteLine("УСЛУГОВАЯ ФАКТУРА");
        base.PrintInfo();
        Console.WriteLine($"Дата оказания услуги: {ServiceDate:dd.MM.yyyy}");
    }
}

public class CombinedInvoice : Invoice
{
    public bool IsMixed { get; set; }
    public decimal ReturnAmount { get; set; }

    public CombinedInvoice(string invoiceNumber, DateTime issueDate, bool isMixed)
        : base(invoiceNumber, issueDate)
    {
        IsMixed = isMixed;
    }
}

Товарная позиция "Ноутбук" добавлена. Дата поставки: 15.09.2026
Товарная позиция "Компьютерная мышь" добавлена. Дата поставки: 15.09.2026
Удаление услуги "Консультация". Причина: Услуга отменена клиентом
ПОЗИЦИИ ДОБАВЛЕНЫ И УДАЛЕНЫ
Проверка завершена.


In [64]:


Invoice invoice1 = goodsInvoice;
Invoice invoice2 = serviceInvoice;
Invoice invoice3 = combinedInvoice;

Console.WriteLine("ПОЛИМОРФИЗМ");

invoice1.PrintInfo();
Console.WriteLine();

invoice2.PrintInfo();
Console.WriteLine();

invoice3.PrintInfo();
Console.WriteLine("КОНЕЦ ");

ПОЛИМОРФИЗМ
 ТОВАРНАЯ ФАКТУРА
Фактура №G-001 от 12.09.2026
Дата поставки: 15.09.2026

УСЛУГОВАЯ ФАКТУРА
Фактура №S-001 от 12.09.2026
Дата оказания услуги: 13.09.2026

Фактура №C-001 от 12.09.2026
КОНЕЦ 


## Результат работы программы

В результате работы программы создаются три разных вида фактур:

1. **GoodsInvoice** - хранит дату поставки и автоматически записывает её в добавляемую позицию.
2. **ServiceInvoice** - хранит дату оказания услуги и при удалении позиции фиксирует причину её аннулирования.
3. **CombinedInvoice** - объединяет товары и услуги и учитывает разрешение на возврат и сумму возврата при расчёте итоговой стоимости.

Таким образом, требования варианта № 10 выполнены. Базовый класс `Invoice` содержит общие свойства и методы, а производные классы расширяют его функциональность и переопределяют методы для демонстрации полиморфизма.

### Основные принципы ООП, использованные в проекте

- **Наследование** - `GoodsInvoice`, `ServiceInvoice` и `CombinedInvoice` наследуются от `Invoice`.
- **Инкапсуляция** - список позиций хранится внутри класса, а итоговая сумма изменяется через методы.
- **Полиморфизм** - переопределённые методы `CalculateTotal()`, `AddLine()`, `RemoveLine()` и `PrintInfo()` вызываются через ссылки базового типа `Invoice`.
- **Расширяемость** - новые виды фактур можно добавлять без изменения основной логики существующих классов.
